# Predicting experimental Curie-Temperatures from compound embeddings 

This pipeline trains machine learning models that predict experimental Curie temperatures (Tc_exp, in Kelvin) directly from stoichiometric compound embeddings — without any simulated Tc values or data augmentation.

In [1]:
import sys
from pathlib import Path
import os

# Get project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent.resolve()

# Add project root to Python path so src/ is importable
sys.path.insert(0, str(PROJECT_ROOT))

# Change working directory to project root
os.chdir(PROJECT_ROOT)

### 1. Pre-Process Data

The preprocessing pipeline in this script prepares experimental and simulated Curie temperature (Tc) data.

1. **Aggregate** data from multiple sources.  
2. **Clean** Tc values: remove units, symbols, and uncertainties; convert to float.  
3. **Drop** invalid (non-numeric) Tc entries.  
4. **Deduplicate** by taking the median Tc per composition.  
5. **Flag** compositions containing rare-earth elements.  
6. **Split** data into RE-containing and RE-free subsets.  
7. **Save** clean, structured datasets for analysis.

In [2]:
from src.process_tc_data import main

In [3]:
main()

Experimental  — total: 14480, duplicates: 0
Simulated     — total: 2107, duplicates: 0


### 2. Embedding Creation

1. **Load Element Embeddings**:  
   Read pre-trained element vectors (e.g., Matscholar200) from a JSON file.

2. **Generate Compound Embeddings**:  
   For each composition, compute a weighted average of its constituent element embeddings, where weights are based on atomic fractions.

3. **Handle Missing Elements**:  
   Skip compounds containing elements not in the embedding dictionary.

4. **Filter Invalid Embeddings**:  
   Remove compounds that couldn’t be embedded (e.g., due to unknown elements).

5. **Save Results**:  
   Store the resulting DataFrame (with embeddings) as a pickle file for each dataset.

In [4]:
from src.create_embeddings import create_embeddings

In [5]:
create_embeddings()

Creating compound embeddings for experimental Tc datasets
Vocabulary: 103 elements  |  dimension: 200

------------------------------------------------------------
Dataset : RE-Free  (Experimental_Tc_RE-Free.csv)
  Rows with valid Tc_exp : 5680
  Dropped (un-embeddable): 34
  Embeddable rows        : 5646

------------------------------------------------------------
Dataset : RE  (Experimental_Tc_RE.csv)
  Rows with valid Tc_exp : 8800
  Dropped (un-embeddable): 19
  Embeddable rows        : 8781

------------------------------------------------------------
Dataset : All  (Experimental_Tc_all.csv)
  Rows with valid Tc_exp : 14480
  Dropped (un-embeddable): 53
  Embeddable rows        : 14427

Done. Next step: python src/compress_embeddings_pca.py


### 3. Compress Embeddings with PCA

1. **Load Pre-Computed Embeddings**:  
   Reads compound embeddings from pickle files (for RE-free, RE-containing, and All experimental datasets).

2. **Apply PCA**:  
   Compresses high-dimensional embeddings (e.g., 200+ dims) into lower-dimensional representations using PCA with 8, 16, 32, and 64 components.

3. **Preserve Explained Variance**:  
   Tracks how much variance each reduced dimension captures (typically >90% with 64 components).

4. **Save Compressed Embeddings**:  
   Adds new columns (`comp_emb_pca_8`, `comp_emb_pca_16`, etc.) to the DataFrame and saves the result as a new pickle file.

In [6]:
from src.compress_embeddings_pca import compress_embeddings_pca

In [7]:
compress_embeddings_pca()

PCA compression of compound embeddings
Component sizes: [8, 16, 32, 64]

------------------------------------------------------------
Dataset : RE-Free
  Loaded 5646 rows
  Raw embeddings shape: (5646, 200)
  PCA( 8 components): explained variance = 0.748  → 'comp_emb_pca_8'
  PCA(16 components): explained variance = 0.884  → 'comp_emb_pca_16'
  PCA(32 components): explained variance = 0.975  → 'comp_emb_pca_32'
  PCA(64 components): explained variance = 1.000  → 'comp_emb_pca_64'

------------------------------------------------------------
Dataset : RE
  Loaded 8781 rows
  Raw embeddings shape: (8781, 200)
  PCA( 8 components): explained variance = 0.819  → 'comp_emb_pca_8'
  PCA(16 components): explained variance = 0.916  → 'comp_emb_pca_16'
  PCA(32 components): explained variance = 0.978  → 'comp_emb_pca_32'
  PCA(64 components): explained variance = 1.000  → 'comp_emb_pca_64'

------------------------------------------------------------
Dataset : All
  Loaded 14427 rows
  Raw emb

### 4. Model Training

1. **Input**:  
   Preprocessed experimental datasets with PCA-compressed compound embeddings (8–64D) and Tc_exp targets.

2. **Models Trained per Dataset** (RE-Free, RE, All):  
   - **Linear** (LassoLars, Ridge)  
   - **Random Forest**  
   - **MLP (Neural Network)**  
   All trained on raw 200D and PCA-reduced embeddings (8, 16, 32, 64D).

3. **Hyperparameter Tuning**:  
   - Randomized search (RF, MLP) and grid search (linear).  
   - Search space scaled to dataset size (e.g., fewer iterations for larger datasets).

4. **Evaluation & Visualization**:  
   - Metrics: R², MAE, RMSE.  
   - Plots: Predicted vs. actual Tc for train/test sets.

5. **Model Export**:  
   - Best models saved as ONNX (with preprocessing included).

6. **Output**:  
   - Per-dataset results (`results/<dataset>_results.csv`).  
   - Global comparison and best model summary (`exp_tc_comparison.csv`, `exp_tc_best_by_dataset.csv`).  
   - Figures and ONNX models for analysis and inference.

In [8]:
from src.train_exp_tc_all import main as main_all

[train_exp_tc] using n_jobs=8 for joblib/loky parallelism


In [9]:
main_all()       # All (combined) dataset

Training (All): compound embedding → experimental Tc

Dataset : All  (Experimental_Tc_all_w_embeddings_PCA.pkl)
Loaded 14427 rows
  RE physics features: ON (7 cols, 8090/14427 rows with RE content)

  [raw_200D]  X: (14427, 207)  train: ~11541  test: ~2886
    (RF search: n_iter=17, cv=3 on 11541 samples)...  

Exception ignored in: <function ResourceTracker.__del__ at 0x1552c55be700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x15224d0c2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/pa

done
    Training RF (ensemble 1/10)...    Figure: All_raw_200D_e0_rf.png
R²=0.8435  RMSE=106.9 K  MAE=59.6 K
  ONNX → All_raw_200D_rf_refeats_e0.onnx
    Training RF (ensemble 2/10)...    Figure: All_raw_200D_e1_rf.png
R²=0.8706  RMSE=97.9 K  MAE=58.0 K
  ONNX → All_raw_200D_rf_refeats_e1.onnx
    Training RF (ensemble 3/10)...    Figure: All_raw_200D_e2_rf.png
R²=0.8604  RMSE=101.8 K  MAE=58.3 K
  ONNX → All_raw_200D_rf_refeats_e2.onnx
    Training RF (ensemble 4/10)...    Figure: All_raw_200D_e3_rf.png
R²=0.8696  RMSE=98.8 K  MAE=58.0 K
  ONNX → All_raw_200D_rf_refeats_e3.onnx
    Training RF (ensemble 5/10)...    Figure: All_raw_200D_e4_rf.png
R²=0.8437  RMSE=105.7 K  MAE=60.8 K
  ONNX → All_raw_200D_rf_refeats_e4.onnx
    Training RF (ensemble 6/10)...    Figure: All_raw_200D_e5_rf.png
R²=0.8536  RMSE=101.8 K  MAE=58.4 K
  ONNX → All_raw_200D_rf_refeats_e5.onnx
    Training RF (ensemble 7/10)...    Figure: All_raw_200D_e6_rf.png
R²=0.8402  RMSE=109.0 K  MAE=61.2 K


Exception ignored in: <function ResourceTracker.__del__ at 0x1507b32b2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x14c4ecdbe700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/pa

  ONNX → All_raw_200D_rf_refeats_e6.onnx
    Training RF (ensemble 8/10)...    Figure: All_raw_200D_e7_rf.png
R²=0.8521  RMSE=103.9 K  MAE=58.8 K
  ONNX → All_raw_200D_rf_refeats_e7.onnx
    Training RF (ensemble 9/10)...    Figure: All_raw_200D_e8_rf.png
R²=0.8537  RMSE=103.0 K  MAE=59.3 K
  ONNX → All_raw_200D_rf_refeats_e8.onnx
    Training RF (ensemble 10/10)...    Figure: All_raw_200D_e9_rf.png
R²=0.8574  RMSE=102.1 K  MAE=58.7 K
  ONNX → All_raw_200D_rf_refeats_e9.onnx
    (LGBM search: n_iter=17, cv=3 on 11541 samples)...  done
    Training LGBM (ensemble 1/10)...    Figure: All_raw_200D_e0_lgbm.png
R²=0.8497  RMSE=104.8 K  MAE=58.4 K
  ONNX → All_raw_200D_lgbm_refeats_e0.onnx
    Training LGBM (ensemble 2/10)...    Figure: All_raw_200D_e1_lgbm.png
R²=0.8764  RMSE=95.7 K  MAE=56.2 K
  ONNX → All_raw_200D_lgbm_refeats_e1.onnx
    Training LGBM (ensemble 3/10)...    Figure: All_raw_200D_e2_lgbm.png
R²=0.8664  RMSE=99.6 K  MAE=56.9 K
  ONNX → All_raw_200D_lgbm_refeats_e2.onnx
    T

In [10]:
from src.train_exp_tc_re import main as main_re

In [11]:
main_re()

Training (RE): compound embedding → experimental Tc

Dataset : RE  (Experimental_Tc_RE_w_embeddings_PCA.pkl)
Loaded 8781 rows
  RE physics features: ON (7 cols, 8090/8781 rows with RE content)

  [raw_200D]  X: (8781, 207)  train: ~7024  test: ~1757
    (RF search: n_iter=28, cv=3 on 7024 samples)...  done
    Training RF (ensemble 1/10)...    Figure: RE_raw_200D_e0_rf.png
R²=0.9157  RMSE=79.3 K  MAE=46.2 K
  ONNX → RE_raw_200D_rf_refeats_e0.onnx
    Training RF (ensemble 2/10)...  

Exception ignored in: <function ResourceTracker.__del__ at 0x149b606b2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  Figure: RE_raw_200D_e1_rf.png
R²=0.9142  RMSE=77.9 K  MAE=45.1 K


Exception ignored in: <function ResourceTracker.__del__ at 0x14f29c8be700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  ONNX → RE_raw_200D_rf_refeats_e1.onnx
    Training RF (ensemble 3/10)...    Figure: RE_raw_200D_e2_rf.png
R²=0.9255  RMSE=72.5 K  MAE=42.5 K
  ONNX → RE_raw_200D_rf_refeats_e2.onnx
    Training RF (ensemble 4/10)...    Figure: RE_raw_200D_e3_rf.png
R²=0.9089  RMSE=82.3 K  MAE=45.0 K
  ONNX → RE_raw_200D_rf_refeats_e3.onnx
    Training RF (ensemble 5/10)...    Figure: RE_raw_200D_e4_rf.png
R²=0.9228  RMSE=75.8 K  MAE=43.7 K


...

  ONNX → RE_raw_200D_rf_refeats_e4.onnx
    Training RF (ensemble 6/10)...    Figure: RE_raw_200D_e5_rf.png
R²=0.9220  RMSE=75.1 K  MAE=44.3 K
  ONNX → RE_raw_200D_rf_refeats_e5.onnx
    Training RF (ensemble 7/10)...    Figure: RE_raw_200D_e6_rf.png
R²=0.9379  RMSE=68.8 K  MAE=41.4 K


Exception ignored in: <function ResourceTracker.__del__ at 0x15108c4c2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  ONNX → RE_raw_200D_rf_refeats_e6.onnx
    Training RF (ensemble 8/10)...  

Exception ignored in: <function ResourceTracker.__del__ at 0x150e926c2700>
Traceback (most recent call last):
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/mpcdf/soft/SLE_15/packages/x86_64/python-waterboa/2025.06/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


  Figure: RE_raw_200D_e7_rf.png
R²=0.9306  RMSE=69.6 K  MAE=41.2 K
  ONNX → RE_raw_200D_rf_refeats_e7.onnx
    Training RF (ensemble 9/10)...    Figure: RE_raw_200D_e8_rf.png
R²=0.9289  RMSE=71.8 K  MAE=42.0 K
  ONNX → RE_raw_200D_rf_refeats_e8.onnx
    Training RF (ensemble 10/10)...    Figure: RE_raw_200D_e9_rf.png
R²=0.9395  RMSE=67.7 K  MAE=42.9 K
  ONNX → RE_raw_200D_rf_refeats_e9.onnx
    (LGBM search: n_iter=28, cv=3 on 7024 samples)...  done
    Training LGBM (ensemble 1/10)...    Figure: RE_raw_200D_e0_lgbm.png
R²=0.9230  RMSE=75.8 K  MAE=42.9 K
  ONNX → RE_raw_200D_lgbm_refeats_e0.onnx
    Training LGBM (ensemble 2/10)...    Figure: RE_raw_200D_e1_lgbm.png
R²=0.9148  RMSE=77.7 K  MAE=42.4 K
  ONNX → RE_raw_200D_lgbm_refeats_e1.onnx
    Training LGBM (ensemble 3/10)...    Figure: RE_raw_200D_e2_lgbm.png
R²=0.9302  RMSE=70.1 K  MAE=39.3 K
  ONNX → RE_raw_200D_lgbm_refeats_e2.onnx
    Training LGBM (ensemble 4/10)...    Figure: RE_raw_200D_e3_lgbm.png
R²=0.9160  RMSE=79.0 K  MAE

In [12]:
from src.train_exp_tc_re_free import main as main_re_free

In [13]:
main_re_free()

Training (RE-Free): compound embedding → experimental Tc

Dataset : RE-Free  (Experimental_Tc_RE-Free_w_embeddings_PCA.pkl)
Loaded 5646 rows
  RE physics features: ON (7 cols, 0/5646 rows with RE content)

  [raw_200D]  X: (5646, 207)  train: ~4516  test: ~1130
    (RF search: n_iter=44, cv=3 on 4516 samples)...  done
    Training RF (ensemble 1/10)...    Figure: RE-Free_raw_200D_e0_rf.png
R²=0.7355  RMSE=134.2 K  MAE=80.3 K
  ONNX → RE-Free_raw_200D_rf_refeats_e0.onnx
    Training RF (ensemble 2/10)...    Figure: RE-Free_raw_200D_e1_rf.png
R²=0.7556  RMSE=131.2 K  MAE=78.9 K
  ONNX → RE-Free_raw_200D_rf_refeats_e1.onnx
    Training RF (ensemble 3/10)...    Figure: RE-Free_raw_200D_e2_rf.png
R²=0.7203  RMSE=139.4 K  MAE=83.3 K
  ONNX → RE-Free_raw_200D_rf_refeats_e2.onnx
    Training RF (ensemble 4/10)...    Figure: RE-Free_raw_200D_e3_rf.png
R²=0.7024  RMSE=141.9 K  MAE=86.2 K
  ONNX → RE-Free_raw_200D_rf_refeats_e3.onnx
    Training RF (ensemble 5/10)...    Figure: RE-Free_raw_200D_e